# KCA extension experiments (full scale, P4Transformer)

> **English copy of the executed notebook.** Text, comments and printed messages were translated from Chinese;
> the code is unchanged and the outputs are the ones recorded in the actual run (Oct 22, 2026, Colab A100).
> The saved copy stops after Section 2: the outputs of Sections 3–4 were not saved with the notebook. Those runs
> did finish; their logs are `code/results/msr_full/logs/crossover_seed{3,4,5}.log`, their results
> `code/results/msr_full/crossover_seed{3,4,5}/crossover.json`, and the summary numbers are in `code/results/RESULTS.md`.

Three experiments, all using the full-scale checkpoints trained on 10/21 (read-only); no backbone is retrained:
1. **Extend the main benchmark to every test video**: it used to be 5 clips per action, 100 in total. The existing 100 records are copied first and the remaining videos are run after them; existing records are not recomputed.
2. **Re-sampling stability (new)**: 40 clips, 5 independent point samplings per clip, comparing how stable each method's explanation maps are.
3. **Crossover with more seeds**: run the P4Transformer crossover again with seeds 3, 4, 5.

**How to use**: choose an A100 runtime → **Runtime → Run all**, and allow Drive access. About 2.5 hours, about 13 compute units. After a disconnect, "Run all" again; finished parts are skipped automatically.

**Old results are not overwritten**: it only reads `kca_full/upload/` (data), `kca_full/ckpt/` (models) and `kca_full/results/` (existing records); the new code bundle and every output are in `kca_full/extend/`.

In [1]:
# ===== Configuration =====
import os, sys, glob, json, shutil, subprocess, time

DRIVE_ROOT = "/content/drive/MyDrive/kca_full"
UPLOAD = f"{DRIVE_ROOT}/upload"          # read-only: MSR point-cloud data
CK = f"{DRIVE_ROOT}/ckpt"                # read-only: the full-scale checkpoints from 10/21
OLD = f"{DRIVE_ROOT}/results/msr"        # read-only: the existing per-clip records
EXT = f"{DRIVE_ROOT}/extend"             # new code bundle + every output of this notebook
POINTS = 2048
SEEDS = [3, 4, 5]                        # new crossover seeds (0, 1, 2 were already run)

In [2]:
# Mount Drive; check the GPU, code bundle, data and checkpoint
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → choose A100"
p = torch.cuda.get_device_properties(0)
print(f"GPU {p.name} {p.total_memory / 1e9:.0f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")
for f in (f"{EXT}/kca_bundle.tar.gz", f"{UPLOAD}/msr_npz.tar.gz", f"{CK}/p4t/last.pth", f"{OLD}/p4transformer/raw.jsonl"):
    assert os.path.exists(f), f"{f} not found (Colab must be signed in to the Google account that holds kca_full/)"
print("code bundle, data, checkpoint, existing records ✓")

Mounted at /content/drive
GPU NVIDIA A100-SXM4-40GB 42 GB | torch 2.11.0+cu130 | CUDA 13.0
code bundle, data, checkpoint, existing records ✓


In [3]:
# Unpack to the local disk, define run()
W = "/content/kca_extend"
CODE, DATA = f"{W}/code", f"{W}/data/msr"
LOGS = f"{EXT}/logs"
os.makedirs(LOGS, exist_ok=True)
if not os.path.exists(f"{CODE}/src/kca/run/msr_stability.py"):
    shutil.rmtree(W, ignore_errors=True)
    os.makedirs(W)
    subprocess.run(["tar", "-xzf", f"{EXT}/kca_bundle.tar.gz", "-C", W], check=True)
if len(glob.glob(f"{DATA}/*.npz")) < 567:
    os.makedirs(f"{W}/data", exist_ok=True)
    subprocess.run(["tar", "-xzf", f"{UPLOAD}/msr_npz.tar.gz", "-C", f"{W}/data"], check=True)
assert len(glob.glob(f"{DATA}/*.npz")) >= 567

PY = sys.executable
ENV = dict(os.environ, PYTHONPATH=f"{CODE}/src", PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
           MPLBACKEND="Agg", KCA_EXT=f"{W}/ext", KCA_PURE_OPS="1")
if os.path.isdir("/usr/local/cuda"):
    ENV.setdefault("CUDA_HOME", "/usr/local/cuda")


def run(cmd, log=None):
    print("$", cmd, flush=True)
    t0, fh = time.time(), (open(f"{LOGS}/{log}", "a") if log else None)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if fh:
                fh.write(line); fh.flush()
        p.wait()
    except KeyboardInterrupt:
        p.kill(); raise
    finally:
        if fh:
            fh.close()
    print(f"[exit {p.returncode} · {(time.time() - t0) / 60:.1f} min]", flush=True)
    if p.returncode:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")


subprocess.run([PY, "-m", "pip", "install", "-q", "einops", "timm", "termcolor", "ninja"], check=True)
r = subprocess.run([PY, "-c", "from kca.backbones import cpu_ops; cpu_ops.install(); "
                    "from kca.backbones import cuda_fast; import json; print(json.dumps(cuda_fast.STATUS))"],
                   cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stderr[-1500:])
print("Fast paths:", r.stdout.strip().splitlines()[-1] if r.returncode == 0 else "check failed (same results, only slower)")

[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: off (ModuleNotFoundError: No module named 'selective_scan_cuda')

Fast paths: {"fps_same_as_loop_m64": 1.0, "fps_same_as_loop_m1024": 1.0, "fps": "on", "scan": "off (ModuleNotFoundError: No module named 'selective_scan_cuda')"}


## 1. Extend the main benchmark to every test video (about 90 minutes)

In [4]:
OUT1 = f"{EXT}/p4transformer"
os.makedirs(OUT1, exist_ok=True)
if not os.path.exists(f"{OUT1}/raw.jsonl"):           # copy the existing 100 records first; they are not recomputed
    shutil.copy(f"{OLD}/p4transformer/raw.jsonl", f"{OUT1}/raw.jsonl")
run(f"{PY} -m kca.run.msr_benchmark --backbone p4transformer --ckpt {CK}/p4t/last.pth --data {DATA} "
    f"--out {OUT1} --per-class 99 --points {POINTS} --device cuda --threads 4", log="bench_p4transformer_all.log")

$ /usr/bin/python3 -m kca.run.msr_benchmark --backbone p4transformer --ckpt /content/drive/MyDrive/kca_full/ckpt/p4t/last.pth --data /content/kca_extend/data/msr --out /content/drive/MyDrive/kca_full/extend/p4transformer --per-class 99 --points 2048 --device cuda --threads 4
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: off (ModuleNotFoundError: No module named 'selective_scan_cuda')
[p4transformer] 287 clips (100 done), methods=['random', 'geometry_edge', 'motion_prior', 'grad_x_input', 'smoothgrad', 'integrated_gradients', 'grad_cam_3d', 'point_drop', 'occam', 'extremal', 'attn_rollout', 'transformer_attr', 'staa4d', 'kca', 'kca_ctx']
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into 

## 2. Re-sampling stability (about 30 minutes)

In [ ]:
run(f"{PY} -m kca.run.msr_stability --backbone p4transformer --ckpt {CK}/p4t/last.pth --data {DATA} "
    f"--out {EXT}/stability --per-class 2 --samplings 5 --points {POINTS} --device cuda --threads 4",
    log="stability.log")

$ /usr/bin/python3 -m kca.run.msr_stability --backbone p4transformer --ckpt /content/drive/MyDrive/kca_full/ckpt/p4t/last.pth --data /content/kca_extend/data/msr --out /content/drive/MyDrive/kca_full/extend/stability --per-class 2 --samplings 5 --points 2048 --device cuda --threads 4
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: off (ModuleNotFoundError: No module named 'selective_scan_cuda')
[stability/p4transformer] 40 clips x 5 samplings (0 done), methods=['random', 'motion_prior', 'grad_x_input', 'integrated_gradients', 'point_drop', 'occam', 'attn_rollout', 'kca', 'kca_ctx']
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
  [1/40] a01_s06_e

## 3. Crossover: P4Transformer, seeds 3, 4, 5 (about 10 minutes each)

In [ ]:
for s in SEEDS:
    out = f"{EXT}/crossover_seed{s}"
    j = f"{out}/crossover.json"
    if os.path.exists(j) and all("kca" in json.load(open(j)).get(t, {}) for t in ("motion", "geometry")):
        print(f"seed {s}: done")
        continue
    run(f"{PY} -m kca.run.msr_crossover --backbone p4transformer --seed {s} --data {DATA} --out {out} "
        f"--scale official --points {POINTS} --device cuda --explain-device cuda --reuse", log=f"crossover_seed{s}.log")

## 4. Summary (copy this cell's output for the analysis)

In [ ]:
import numpy as np
n = sum(1 for l in open(f"{EXT}/p4transformer/raw.jsonl") if l.strip())
print(f"1. Main benchmark: {n} clips")
recs = [json.loads(l) for l in open(f"{EXT}/stability/stability.jsonl") if l.strip()]
print(f"2. Stability: {len(recs)} clips × {recs[0]['samplings']} samplings; rank correlation with sampling 0 (point level / part level / point level after averaging two / part level after averaging two)")
for m in ("integrated_gradients", "point_drop", "kca", "kca_ctx"):
    v = [np.nanmean([np.nanmean(r[k][m]) if isinstance(r[k].get(m), list) else r[k].get(m, np.nan) for r in recs])
         for k in ("rank_corr", "part_corr", "avg2_rank_corr", "avg2_part_corr")]
    print(f"   {m:22s} " + "  ".join(f"{x:.2f}" for x in v))
print("3. Crossover seeds:")
for s in SEEDS:
    c = json.load(open(f"{EXT}/crossover_seed{s}/crossover.json"))
    am, ag = 100 * c["motion"]["test_acc"], 100 * c["geometry"]["test_acc"]
    gap = c.get("ci_kca", {}).get("gap", [float("nan")] * 2)
    flag = "✓" if min(am, ag) >= 60 else "⚠️ not learned (< 60%)"
    print(f"   seed {s}: accuracy {am:.1f} / {ag:.1f}% | motion share {c['motion']['kca']['arm_motion_share']:.2f} / "
          f"{c['geometry']['kca']['arm_motion_share']:.2f} | gap CI [{gap[0]:.2f}, {gap[1]:.2f}] {flag}")